# 01. Primeiras chamadas ao modelo

Uma LLM (modelo de linguagem de grande porte) recebe texto e gera uma resposta. Nesta aula, vamos conversar com um modelo que roda na nossa máquina pelo Ollama.

Execute uma célula por vez, de cima para baixo. Vamos configurar a conexão, enviar uma pergunta, observar o objeto retornado e só então extrair o texto. Depois, veremos instruções, temperatura e streaming.

## 1. Prepare a conexão com o modelo

`ChatOllama` é a classe do LangChain que usamos para conversar com o Ollama. A importação disponibiliza essa classe no Python.


In [32]:
from langchain_ollama import ChatOllama

Agora criamos `model`, o objeto que usaremos para fazer as chamadas:

- `base_url`: endereço do Ollama em execução nesta máquina.
- `model`: nome do modelo já baixado na preparação.
- `temperature`: controla a variação na geração; começaremos com `0`.
- `reasoning=False`: desativa o modo de raciocínio separado neste exemplo.

Esta célula configura o cliente. A pergunta será enviada apenas quando chamarmos `invoke`.


In [22]:
model = ChatOllama(
    base_url="http://127.0.0.1:11434",
    model="qwen3.5:4b",
    temperature=0,
    reasoning=False,
)

## 2. Envie a primeira pergunta

O texto enviado ao modelo também é chamado de **prompt**. `invoke` envia esse texto e espera a resposta completa. Guardamos o retorno na variável `response` para examiná-lo depois. A chamada pode levar alguns segundos.


In [33]:
response = model.invoke("Qual a capital do Brasil?")


Antes de selecionar qualquer campo, vamos imprimir o retorno inteiro. Observe que ele traz mais informações do que a frase que responde é pergunta.


In [34]:
print(response)

content='A capital do Brasil é **Brasília**.\n\nLocalizada no interior da região central do país, Brasília foi escolhida como sede para equilibrar o desenvolvimento entre as regiões Norte e Sul, além de evitar a concentração excessiva de poder em São Paulo. A cidade foi inaugurada em 1960, substituindo Rio de Janeiro como capital federal.' additional_kwargs={} response_metadata={'model': 'qwen3.5:4b', 'created_at': '2026-10-28T17:15:07.473340828Z', 'done': True, 'done_reason': 'stop', 'total_duration': 16288177134, 'load_duration': 14202237493, 'prompt_eval_count': 18, 'prompt_eval_duration': 478231000, 'eval_count': 75, 'eval_duration': 1603245000, 'logprobs': None, 'model_name': 'qwen3.5:4b', 'model_provider': 'ollama'} id='lc_run--01a18381-e7bd-71f5-8619-95861245fb21-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 18, 'output_tokens': 75, 'total_tokens': 93}


O retorno é um objeto de mensagem do LangChain. Vamos conferir seu tipo:


In [35]:
print(type(response))

<class 'langchain_core.messages.ai.AIMessage'>


Você deve encontrar o tipo `AIMessage`, que representa uma mensagem produzida pelo modelo. Na impressóo anterior, procure:

- `content`: conteúdo da resposta; neste exemplo, o texto gerado.
- `response_metadata`: informações da execução, que podem incluir o nome do modelo e tempos de processamento.
- `usage_metadata`: informações de uso, quando disponíveis, como a contagem de tokens. Tokens sóo pequenas unidades de texto processadas pelo modelo.

Os valores e campos exibidos podem variar com a versóo e o modelo. Não precisamos interpretar todos agora: primeiro, localize o `content`.

## 4. Acesse somente o texto

Em Python, o ponto permite acessar um atributo do objeto. Assim, `response.content` seleciona o conteúdo da mesma resposta que acabamos de inspecionar.


In [26]:
print(response.content)

A capital do Brasil é **Brasília**.

Localizada no interior da região central do país, Brasília foi escolhida como sede para equilibrar o desenvolvimento entre as regiões Norte e Sul, além de evitar a concentração excessiva de poder em São Paulo. A cidade foi inaugurada em 1960, substituindo Rio de Janeiro como capital federal.


Compare as duas impressóes: `print(response)` mostra a mensagem com seus campos; `print(response.content)` mostra apenas o conteúdo. Não fizemos uma nova chamada ao modelo para acessar esse atributo.

## 5. Mensagens e instruções

Até aqui, enviamos uma pergunta como texto simples. Também podemos enviar uma lista de mensagens, cada uma com um papel:

- `SystemMessage`: define instruções gerais, como o papel do tutor e o estilo da resposta.
- `HumanMessage`: contém a pergunta do usuário.

Vamos importar essas classes e montar a lista antes de chamar o modelo.


In [37]:
from langchain.messages import SystemMessage, HumanMessage

In [38]:
messages = [
    SystemMessage(content="Você é um tutor de ciências. Responda em português de forma breve."),
    HumanMessage(content="O que é erosão?"),
]

Leia a lista antes de executar a próxima célula: qual mensagem define o comportamento do tutor? Qual traz a pergunta? O parâmetro `content` guarda o texto de cada mensagem.

Agora enviamos a lista em ordem. O retorno continua sendo uma mensagem, e usamos `.content` para exibir seu texto.


In [39]:
response = model.invoke(messages)
print(response.content)

A **erosão** é o processo natural de desgaste e remoção de partículas do solo ou rochas, geralmente causado pela ação da água, vento, gelo ou gravidade. Esse fenômeno transporta os materiais para outras áreas, podendo levar à formação de vales, cânions e ao assoreamento de rios.


## 6. Desafio: compare as temperaturas

A temperatura controla a variação na escolha dos próximos tokens. Temperaturas menores tendem a produzir respostas mais previsóveis; maiores permitem mais variação. Isso não garante respostas iguais nem corretas.

A célula abaixo repete a mesma pergunta com `temperature=0` e `temperature=1`. Em cada volta do `for`, criamos um cliente com a temperatura escolhida, fazemos a chamada e imprimimos o conteúdo da resposta.

Execute algumas vezes e compare: as analogias mudam? Elas explicam o ciclo da água corretamente? Uma resposta fluente também precisa ser conferida.


In [41]:
for temperature in (0, 5):
    comparison_model = ChatOllama(
        base_url="http://127.0.0.1:11434",
        model="qwen3.5:4b",
        temperature=temperature,
        reasoning=False,
    )
    response = comparison_model.invoke("Crie uma analogia para o ciclo da água. Responda em português.")
    print("\nTEMPERATURA", temperature)
    print(response.content)


TEMPERATURA 0
Uma boa analogia para o ciclo da água é a de um **grande sistema de irrigação automático** ou, mais visualmente, como um **jogo de "pegar e soltar" entre dois amigos**.

Imagine que você tem uma piscina cheia (o oceano) e um amigo está sentado em uma cadeira ao lado (a terra). O ciclo funciona assim:

1.  **Evaporação:** Você joga água da piscina para cima, fazendo gotas subirem no ar.
2.  **Condensação:** As gotas se juntam no ar até formar nuvens densas e escuras.
3.  **Precipitação:** De repente, as gotas ficam tão pesadas que caem de volta na piscina (chuva ou neve).
4.  **Infiltração/Escoamento:** Parte da água cai direto de volta no seu amigo (rios e oceanos), enquanto outra parte vai para o chão dele, onde ele pode beber ou usar para regar plantas.

O ponto chave é que **nada se perde nem é criado**; a mesma quantidade de água continua circulando infinitamente, apenas mudando de lugar e de estado (líquido, gasoso ou sólido), exatamente como as gotas continuam viaj

## 7. Receba a resposta em partes

`invoke` espera a resposta completa. Com `stream`, recebemos trechos conforme sóo gerados, em uma única chamada.

Vamos reutilizar `model`, que já está configurado. No laço abaixo, `chunk` é cada parte recebida, e `chunk.content` é seu conteúdo. `end=""` evita uma quebra de linha entre os trechos; `flush=True` pede que a saída seja exibida imediatamente.

Execute e observe o texto aparecendo aos poucos. Um trecho não corresponde necessariamente a uma palavra inteira, e alguns podem ter conteúdo vazio.


In [42]:
for chunk in model.stream("Explique em um parágrafo por que ocorrem as estações do ano. Responda em português."):
    print(chunk.content, end="", flush=True)
print()

As estações do ano ocorrem devido à inclinação do eixo da Terra, que forma um ângulo de aproximadamente 23,5 graus em relação ao plano da sua órbita ao redor do Sol; enquanto a Terra gira em torno do Sol, essa inclinação faz com que diferentes hemisférios recebam quantidades variadas de radiação solar ao longo do ano. Quando o Hemisfério Sul está inclinado em direção ao Sol, ele experimenta o verão e recebe mais luz direta, enquanto o Hemisfério Norte está inclinado para longe, sofrendo o inverno; esse fenômeno se inverte seis meses depois quando a Terra completa sua órbita, fazendo com que o Hemisfério Norte receba a maior incidência solar e entre no verão, enquanto o Sul entra no inverno.


### O modelo lembra das chamadas anterioresó

Cada chamada recebe apenas as mensagens enviadas nela. Neste notebook, ainda não guardamos nem reenviamos o histórico. No próximo, vamos fazer isso com uma lista Python.